In [2]:
import xml.etree.ElementTree as ET
import json
import re

def clean_label(raw):
    # Convert sets to readable labels
    if raw.startswith("({'"):
        try:
            parts = eval(raw)  # Convert to Python set
            return f"{', '.join(sorted(parts[0]))} → {', '.join(sorted(parts[1]))}"
        except:
            return raw
    return raw

def pnml_to_d3_json(pnml_path, method):
    tree = ET.parse(pnml_path)
    root = tree.getroot()

    net = root.find(".//net")
    page = net.find("page")

    nodes = []
    edges = []

    for place in page.findall("place"):
        pid = place.attrib["id"]
        label_node = place.find("name/text")
        label = label_node.text if label_node is not None else pid
        nodes.append({"id": pid, "type": "place", "label": clean_label(label)})

    for trans in page.findall("transition"):
        tid = trans.attrib["id"]
        label_node = trans.find("name/text")
        label = label_node.text if label_node is not None else tid
        nodes.append({"id": tid, "type": "transition", "label": label})

    for arc in page.findall("arc"):
        src = arc.attrib["source"]
        tgt = arc.attrib["target"]
        edges.append({"source": src, "target": tgt})

    return {
        "method": method,
        "nodes": nodes,
        "edges": edges
    }

# Example use:
alpha = pnml_to_d3_json("data_alpha.pnml", "alpha")
with open("alpha.json", "w") as f:
    json.dump(ilp, f, indent=2)


In [4]:
import xml.etree.ElementTree as ET
import json

def xes_to_trace_list(xes_path):
    tree = ET.parse(xes_path)
    root = tree.getroot()

    trace_list = []

    for trace in root.findall('trace'):
        event_labels = []
        for event in trace.findall('event'):
            for string in event.findall('string'):
                if string.attrib.get('key') == 'concept:name':
                    event_labels.append(string.attrib.get('value'))
                    break
        if event_labels:
            trace_list.append(event_labels)

    return trace_list

def xes_to_d3_json(xes_path, output_json_path):
    traces = xes_to_trace_list(xes_path)
    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(traces, f, indent=2)
    print(f"[✓] Exported {len(traces)} traces to {output_json_path}")
    return traces

# Use like this
if __name__ == "__main__":
    xes_to_d3_json("data.xes", "data.json")


[✓] Exported 1050 traces to data.json


In [2]:
import xml.etree.ElementTree as ET
from collections import defaultdict

pnml_file = 'data_split_reduced.pnml'

tree = ET.parse(pnml_file)
root = tree.getroot()
ns = {'pnml': root.tag.split('}')[0].strip('{')}

# --- Extract places and transitions ---
places = set()
transitions = {}
for place in root.findall('.//pnml:place', ns):
    places.add(place.attrib['id'])

for trans in root.findall('.//pnml:transition', ns):
    label_el = trans.find('./pnml:name/pnml:text', ns)
    label = label_el.text if label_el is not None else ""
    # Exclude invisible transitions (label is $invisible$ or t*)
    if label and not label.startswith('t') and label != '$invisible$':
        transitions[trans.attrib['id']] = label

print("Visible transitions:", transitions)

# --- Extract arcs ---
arcs = []
for arc in root.findall('.//pnml:arc', ns):
    arcs.append({
        'source': arc.attrib['source'],
        'target': arc.attrib['target']
    })

# --- Build input/output maps ---
place_inputs = defaultdict(list)
place_outputs = defaultdict(list)
for arc in arcs:
    if arc['source'] in transitions and arc['target'] in places:
        place_inputs[arc['target']].append(arc['source'])
    if arc['source'] in places and arc['target'] in transitions:
        place_outputs[arc['source']].append(arc['target'])

# --- Find directly-follows edges between visible transitions ---
edges_set = set()
for place_id, outs in place_outputs.items():
    ins = place_inputs[place_id]
    for src in ins:
        for tgt in outs:
            if src in transitions and tgt in transitions:
                edges_set.add((transitions[src], transitions[tgt]))

import json
dfg_json = {
    "nodes": [{"id": name} for name in set(transitions.values())],
    "edges": [{"source": src, "target": tgt} for (src, tgt) in edges_set]
}

with open("dfg.json", "w") as f:
    json.dump(dfg_json, f, indent=2)

print("DFG nodes:", dfg_json["nodes"])
print("DFG edges:", dfg_json["edges"])


Visible transitions: {}
DFG nodes: []
DFG edges: []


In [6]:
import pandas as pd

# If your data is in a .csv or .tsv file:
df = pd.read_csv("sepsis_cases_2_Trunc13.csv", sep="\t")

# Rename columns for clarity
df.columns = ["case", "activity"]

# Create the list of dicts
rawLog = [{"case": row["case"], "activity": row["activity"]} for _, row in df.iterrows()]

# Print as JS array
print("const rawLog = [")
for event in rawLog:
    print(f'  {{case: "{event["case"]}", activity: "{event["activity"]}"}},')
print("];")


ValueError: Length mismatch: Expected axis has 1 elements, new values have 2 elements

: 